In [ ]:
import os
import glob
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import cv2
from sklearn.model_selection import train_test_split


In [ ]:
# Configurations
CFG = {
    "IMG_SIZE": 256,
    "BATCH_SIZE": 8,
    "EPOCHS": 100,
    "LR": 1e-4,
    "DATA_DIR": "./openEDS",
    "SEED": 42,
    "MAX_ROTATION_DEG": 15
}

tf.random.set_seed(CFG["SEED"])
np.random.seed(CFG["SEED"])
AUTOTUNE = tf.data.AUTOTUNE

# Mixed Precision
keras.mixed_precision.set_global_policy('mixed_float16')

IMG_SIZE = CFG["IMG_SIZE"]


In [ ]:

# CLAHE function
def clahe_func(image):
    image = image.astype(np.uint8)
    lab = cv2.cvtColor(image, cv2.COLOR_RGB2LAB)
    l, a, b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    cl = clahe.apply(l)
    lab = cv2.merge([cl, a, b])
    enhanced = cv2.cvtColor(lab, cv2.COLOR_LAB2RGB)
    return enhanced.astype(np.uint8)

# Rotation augmentation function
def rotate_func(image, mask):
    image = (image * 255).astype(np.uint8)
    mask = (mask * 255).astype(np.uint8)
    angle_deg = np.random.uniform(-CFG["MAX_ROTATION_DEG"], CFG["MAX_ROTATION_DEG"])
    h, w = image.shape[:2]
    center = (w // 2, h // 2)
    M = cv2.getRotationMatrix2D(center, angle_deg, 1.0)
    image_rot = cv2.warpAffine(image, M, (w, h), flags=cv2.INTER_LINEAR)
    mask_rot = cv2.warpAffine(mask, M, (w, h), flags=cv2.INTER_NEAREST)
    return (image_rot.astype(np.float32) / 255.0), (mask_rot.astype(np.float32) / 255.0)

# Load .npy mask
def _load_npy(path):
    arr = np.load(path.numpy().decode('utf-8'))
    if arr.ndim == 2:
        arr = arr[..., np.newaxis]
    return arr.astype(np.float32)

# Data Loading
def load_image_mask(img_path, mask_path):
    # Load image from .png
    img = tf.io.read_file(img_path)
    img = tf.image.decode_png(img, channels=3)
    # Load mask from .npy via py_function
    mask = tf.py_function(_load_npy, [mask_path], tf.float32)
    # Set shape so tf.image.resize knows the tensor rank
    mask.set_shape([None, None, 1])
    return img, mask

def preprocess(image, mask):
    image = tf.cast(image, tf.float32) / 255.0
    # Normalize mask: if values > 1 assume 0-255 range
    mask_max = tf.reduce_max(mask)
    mask = tf.cond(mask_max > 1.0, lambda: mask / 255.0, lambda: mask)
    image = tf.image.resize(image, [IMG_SIZE, IMG_SIZE])
    mask = tf.image.resize(mask, [IMG_SIZE, IMG_SIZE], method='nearest')
    mask.set_shape([IMG_SIZE, IMG_SIZE, 1])
    image_uint8 = tf.cast(image * 255, tf.uint8)
    image_clahe = tf.py_function(clahe_func, [image_uint8], tf.uint8)
    image_clahe.set_shape([IMG_SIZE, IMG_SIZE, 3])
    image = tf.cast(image_clahe, tf.float32) / 255.0
    return image, mask

def augment(image, mask):
    image, mask = tf.py_function(rotate_func, [image, mask], [tf.float32, tf.float32])
    image.set_shape([IMG_SIZE, IMG_SIZE, 3])
    mask.set_shape([IMG_SIZE, IMG_SIZE, 1])
    return image, mask

def create_dataset(img_paths, mask_paths, training=True):
    ds = tf.data.Dataset.from_tensor_slices((img_paths, mask_paths))
    ds = ds.map(load_image_mask, num_parallel_calls=AUTOTUNE)
    ds = ds.map(preprocess, num_parallel_calls=AUTOTUNE)
    if training:
        ds = ds.map(augment, num_parallel_calls=AUTOTUNE)
    ds = ds.batch(CFG["BATCH_SIZE"])
    if training:
        ds = ds.shuffle(1000)
    ds = ds.prefetch(AUTOTUNE)
    return ds

def get_image_mask_paths(data_dir):
    # Scan all S_*/*.png for images, pair with corresponding .npy masks
    img_paths = sorted(glob.glob(os.path.join(data_dir, "S_*", "*.png")))
    mask_paths = [p.replace('.png', '.npy') for p in img_paths]
    # Keep only pairs where the .npy mask exists
    valid = [(ip, mp) for ip, mp in zip(img_paths, mask_paths) if os.path.exists(mp)]
    img_paths = [v[0] for v in valid]
    mask_paths = [v[1] for v in valid]
    print(f"Found {len(img_paths)} image/mask pairs from {data_dir}")
    return img_paths, mask_paths


In [ ]:

# Load and split data
all_img_paths, all_mask_paths = get_image_mask_paths(CFG["DATA_DIR"])
train_img_paths, val_img_paths, train_mask_paths, val_mask_paths = train_test_split(
    all_img_paths, all_mask_paths, test_size=0.2, random_state=CFG["SEED"]
)
train_ds = create_dataset(train_img_paths, train_mask_paths, training=True)
val_ds = create_dataset(val_img_paths, val_mask_paths, training=False)


In [ ]:

# Model
def conv_block(inputs, num_filters):
    x = layers.Conv2D(num_filters, 3, padding="same")(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)
    x = layers.Conv2D(num_filters, 3, padding="same")(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)
    return x

def encoder_block(inputs, num_filters):
    x = conv_block(inputs, num_filters)
    p = layers.MaxPool2D((2, 2))(x)
    return x, p

def decoder_block(inputs, skip_features, num_filters):
    x = layers.Conv2DTranspose(num_filters, (2, 2), strides=2, padding="same")(inputs)
    x = layers.Concatenate()([x, skip_features])
    x = conv_block(x, num_filters)
    return x

def build_unet(input_shape):
    inputs = layers.Input(input_shape)
    # Encoder
    s1, p1 = encoder_block(inputs, 64)
    s2, p2 = encoder_block(p1, 128)
    s3, p3 = encoder_block(p2, 256)
    s4, p4 = encoder_block(p3, 512)
    # Bottleneck
    b1 = conv_block(p4, 1024)
    # Decoder
    d1 = decoder_block(b1, s4, 512)
    d2 = decoder_block(d1, s3, 256)
    d3 = decoder_block(d2, s2, 128)
    d4 = decoder_block(d3, s1, 64)
    outputs = layers.Conv2D(1, 1, padding="same", activation="sigmoid")(d4)
    return keras.Model(inputs, outputs)

model = build_unet((IMG_SIZE, IMG_SIZE, 3))


In [ ]:

# Loss and metrics
def dice_coef(y_true, y_pred, smooth=1e-6):
    y_true = tf.cast(y_true, tf.float32)
    y_pred = tf.cast(y_pred, tf.float32)
    intersection = tf.reduce_sum(y_true * y_pred, axis=[1,2,3])
    union = tf.reduce_sum(y_true, axis=[1,2,3]) + tf.reduce_sum(y_pred, axis=[1,2,3])
    return tf.reduce_mean((2. * intersection + smooth) / (union + smooth))

def dice_loss(y_true, y_pred):
    return 1 - dice_coef(y_true, y_pred)

def total_loss(y_true, y_pred):
    bce = keras.losses.binary_crossentropy(y_true, y_pred)
    return bce + dice_loss(y_true, y_pred)


In [ ]:

# Compile
optimizer = keras.optimizers.Adam(learning_rate=CFG["LR"])
model.compile(
    optimizer=optimizer,
    loss=total_loss,
    metrics=[dice_coef, "binary_accuracy"]
)

# Callbacks
callbacks = [
    keras.callbacks.ModelCheckpoint("best_unet_openeds.h5", save_best_only=True, monitor="val_dice_coef", mode="max"),
    keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=5),
    keras.callbacks.EarlyStopping(monitor="val_loss", patience=10, restore_best_weights=True)
]

# Training
history = model.fit(
    train_ds,
    epochs=CFG["EPOCHS"],
    validation_data=val_ds,
    callbacks=callbacks
)

print("Training completed.")